# all-reduce-compose — ex2: compose all_reduce with MAX op via reduce + broadcast

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-compose`. Running the final beacon cell reports progress against the `Distributed: all_reduce composition` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-compose`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-compose"
DD_SUBTOPIC = "Distributed: all_reduce composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `all_reduce` composition with MAX (not SUM)

Ex1 composed `all_reduce(SUM) = reduce(SUM) + broadcast`. The composition is OP-PARAMETRIC — the exact same shape works for MAX, MIN, PRODUCT:

```python
def all_reduce_max(tensor, rank, world_size):
    dist.reduce(tensor, dst=0, op=dist.ReduceOp.MAX)   # rank-0 gets the max
    dist.broadcast(tensor, src=0)                       # everyone learns it
```

**Where MAX-all-reduce shows up.** Global max-norm gradient clipping, early-stop criteria ('any rank diverged?'), max sequence length per batch in dynamic batching. SUM is the most common, but MAX is non-negligible in real DDP code.

**`dist.ReduceOp` is just an enum.** `SUM`, `PRODUCT`, `MAX`, `MIN`, `BAND`, `BOR`, `BXOR`, `PREMUL_SUM`. The reducer dispatches internally — your wrapper only changes the `op=` kwarg.

### Exercise 2 — compose all_reduce with MAX op via reduce + broadcast

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `reduce(MAX) → broadcast` composition to build a custom all-reduce that gives every rank the global maximum across rank-local tensors.
> Keywords: all_reduce, MAX, reduce, broadcast, composition
> ```

**KCs targeted:** `all-reduce-max-via-reduce-broadcast`, `reduce-op-parametric`

Implement `ex2_all_reduce_max(rank, world_size, dist_module, local_value)`. The same `reduce + broadcast` composition shape as ex1, but with the MAX op:

1. Build a 1-D tensor wrapping the rank-local value: `tensor = t.tensor([local_value], dtype=t.float32)`.
2. Reduce to rank 0 with the MAX op: `dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.MAX)`. After this, only rank 0's tensor holds the global maximum.
3. Broadcast from rank 0 so every rank sees it: `dist_module.broadcast(tensor, src=0)`.
4. Return `tensor.item()` — the global maximum, identical on every rank.

Note the signature change vs ex1: this drill takes `dist_module` as a parameter (injected by the test harness — a real `dist` on GPU machines, a mock on CPU). Inside the function, ALL calls go via `dist_module.*` instead of the global `dist.*`. This is a common testing pattern — dependency injection makes the function verifiable on CPU.

Input: `rank`, `world_size` — ints; `dist_module` — the torch.distributed module (or a mock); `local_value` — float.
Output: `float` — the global max, same on every rank.

The test simulates `world_size` ranks via threads and a fake `dist` module that implements `reduce` / `broadcast` / `ReduceOp` with thread-barrier synchronization.

In [ ]:
def ex2_all_reduce_max(rank: int, world_size: int, dist_module, local_value: float) -> float:
    """Compose all_reduce(MAX) from reduce(MAX) + broadcast; return global max."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import contextlib
    import types as _types
    from unittest.mock import patch
    import torch as _t_for_fake
    import torch.distributed as _dist_real

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            # scratch[op_id] -> list of (rank, tensor); reset per op via barrier
            self.scratch = {}
            # per-rank thread-local pinned rank
            self.tls = threading.local()
            # collected per-rank results (for the test to read)
            self.results = [None] * world_size
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            # phase 1: every rank deposits its tensor copy
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            # phase 2: every rank reads-out the reduced result (same math)
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            # mutate in-place so caller's tensor reflects the reduction
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            # only the dst rank gets the reduced result
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            # Build the fake `dist` module facade.
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.init_process_group = lambda **kw: None
            fake_dist.destroy_process_group = lambda: None
            # Inject into the worker's calling globals.
            # The student code calls `dist.<op>`; we patch the `dist` name
            # in the calling namespace via direct globals injection.
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world.results


    # Per-rank local values: rank r has value (1 + r) * 2.5 → [2.5, 5.0, 7.5, 10.0].
    # Global MAX → 10.0 on every rank.
    def _worker(rank, world_size, dist_module, world):
        local_value = (rank + 1) * 2.5
        result = ex2_all_reduce_max(rank, world_size, dist_module, local_value)
        world.results[rank] = result

    results = _run_fake_world(_worker, 4)
    expected_max = 10.0
    for rank, r in enumerate(results):
        assert r is not None, f'rank {rank} returned None — function did not complete'
        assert abs(r - expected_max) < 1e-5, f'rank {rank}: got {r}, expected {expected_max}'

    # Negative values — MAX is sign-aware.
    def _worker_neg(rank, world_size, dist_module, world):
        local_value = -float(rank + 1)   # [-1, -2, -3]
        world.results[rank] = ex2_all_reduce_max(rank, world_size, dist_module, local_value)

    results_neg = _run_fake_world(_worker_neg, 3)
    expected_neg_max = -1.0   # the LEAST negative
    for rank, r in enumerate(results_neg):
        assert abs(r - expected_neg_max) < 1e-5, f'neg: rank {rank}: got {r}, expected {expected_neg_max}'

    # Same value on every rank — global max is that value.
    def _worker_same(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_reduce_max(rank, world_size, dist_module, 42.0)

    results_same = _run_fake_world(_worker_same, 5)
    for rank, r in enumerate(results_same):
        assert abs(r - 42.0) < 1e-5, f'identical-values case rank {rank}: got {r}'

    # Single-rank world — degenerate, max = own value.
    def _worker_single(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_reduce_max(rank, world_size, dist_module, 7.5)

    results_single = _run_fake_world(_worker_single, 1)
    assert abs(results_single[0] - 7.5) < 1e-5
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_all_reduce_max(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.MAX)
    dist_module.broadcast(tensor, src=0)
    return tensor.item()
```

**The composition is `op`-parametric.** Same `reduce → broadcast` shape for SUM, MAX, MIN, PRODUCT — only the `op=` kwarg changes. This is why ARENA teaches the composition: once you know it, every `all_reduce_*` variant slots in.

**Why `dist_module` as a parameter.** Tests on CPU can't run real `gloo`/`nccl` (no fork on Windows, no GPUs on Colab CPU runtimes). Injecting the dist module lets the test pass in a mock that simulates `world_size` ranks via threads + barriers. Production code re-binds `dist_module = torch.distributed` at the call site.

**`reduce(MAX)` + `broadcast` vs real `all_reduce(MAX)`.** Functionally identical. Real `all_reduce` uses tree-reduction (`O(log world_size)` rounds) while compose uses linear (`O(2 * world_size)`). For learning, compose. For production, `dist.all_reduce` directly.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()